In [6]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Load variables from .env into environment
load_dotenv("../../../")

# Retrieve variables
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_name = os.getenv("DB_NAME")

print(f"Loaded API key: {db_name}")

Loaded API key: retail_analytics


In [7]:
import urllib.parse
from sqlalchemy import create_engine, text

# URL-encode passwords containing special characters (e.g., '@', ':', '/')
password = urllib.parse.quote_plus(db_password)
HOST = "localhost"
PORT = "3306"  # Default MySQL port

# URL connection string
# Format: mysql+pymysql://<user>:<password>@<host>:<port>/<database>
DATABASE_URL = f"mysql+pymysql://{db_user}:{password}@{HOST}:{PORT}/{db_name}"

engine = create_engine(
    DATABASE_URL,
    # Enforces the modern, complete UTF-8 charset recommended by SQLAlchemy
    connect_args={"charset": "utf8mb4"}, 
    # Recycles connections older than 1 hour to prevent MySQL "Server has gone away" errors
    pool_recycle=3600,
    # The no of connections to keep open inside the connection pool
    pool_size=10,
    # Max overflow connections allowed beyond the pool_size
    max_overflow=20
)

# Testing connection
with engine.connect() as connection:
    query = text("SELECT DATABASE();")
    result = connection.execute(query)
    
    # Fetch the single row scalar value
    current_db = result.scalar()
    
    print(f"Connected to database: {current_db}")

query = text("""
    SELECT
    cs.customer_id,
    cs.days_since_last_purchase  AS recency,
    cod.order_days               AS frequency,
    cs.total_spend               AS monetary,
    cs.first_purchase_date,
    cs.customer_lifetime_days,
    cs.avg_order_value,
    c.country
    FROM customer_summary cs
    JOIN customer_order_days cod ON cod.customer_id = cs.customer_id
    JOIN customers c             ON c.customer_id   = cs.customer_id;"""
)
rfm = pd.read_sql(query, con=engine)
print(f"DataFrame shape: {rfm.shape}")
rfm.head()
rfm.info()
rfm.describe()


Connected to database: retail_analytics
DataFrame shape: (5852, 8)
<class 'pandas.DataFrame'>
RangeIndex: 5852 entries, 0 to 5851
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             5852 non-null   int64  
 1   recency                 5852 non-null   int64  
 2   frequency               5852 non-null   int64  
 3   monetary                5852 non-null   float64
 4   first_purchase_date     5852 non-null   object 
 5   customer_lifetime_days  5852 non-null   int64  
 6   avg_order_value         5852 non-null   float64
 7   country                 5852 non-null   str    
dtypes: float64(2), int64(4), object(1), str(1)
memory usage: 442.4+ KB


,customer_id,recency,frequency,monetary,customer_lifetime_days,avg_order_value
count,5852.000000,5852.000000,5852.000000,5852.000000,5852.000000,5852.000000
mean,15319.707109,200.732399,5.618250,2916.706762,273.807929,384.405472
std,1714.929942,208.523287,9.916427,14306.854987,258.645596,1254.730667
min,12346.000000,1.000000,1.000000,2.950000,0.000000,2.950000
25%,13837.750000,26.000000,1.000000,339.575000,0.000000,176.142500
50%,15320.500000,96.000000,3.000000,856.020000,223.000000,277.870000
75%,16802.250000,380.000000,6.000000,2241.030000,512.000000,410.557500
max,18287.000000,739.000000,247.000000,580987.040000,738.000000,84236.250000


In [12]:
# 1. Score R, F, and M from 1 to 5 (5 = best, 1 = worst)
rfm["r_score"] = pd.qcut(rfm["recency"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# 2. Combined RFM string and total score
rfm["rfm_code"] = (
    rfm["r_score"].astype(str)
    + rfm["f_score"].astype(str)
    + rfm["m_score"].astype(str)
)
rfm["rfm_total"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

# 3. Sanity checks
print("Null scores:", rfm[["r_score", "f_score", "m_score"]].isnull().sum().to_dict())
print("\nDirection check (mean values by score):")
print("Recency by r_score:\n", rfm.groupby("r_score")["recency"].mean().round(1))
print("\nFrequency by f_score:\n", rfm.groupby("f_score")["frequency"].mean().round(2))
print("\nMonetary by m_score:\n", rfm.groupby("m_score")["monetary"].mean().round(2))
print(f"\nTop '555' customers: {(rfm['rfm_code'] == '555').sum():,}")

Null scores: {'r_score': 0, 'f_score': 0, 'm_score': 0}

Direction check (mean values by score):
Recency by r_score:
 r_score
1    542.8
2    308.8
3    107.5
4     37.1
5      9.8
Name: recency, dtype: float64

Frequency by f_score:
 f_score
1     1.00
2     1.57
3     2.87
4     5.37
5    17.28
Name: frequency, dtype: float64

Monetary by m_score:
 m_score
1      162.24
2      419.32
3      873.86
4     1875.34
5    11248.02
Name: monetary, dtype: float64

Top '555' customers: 473
